In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score,mean_absolute_error
from sklearn.ensemble import AdaBoostRegressor,GradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

In [2]:
df = pd.read_csv('houseprice.csv')

In [3]:
df.shape

(1460, 81)

In [4]:
df.describe()

,Id,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,...,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold,SalePrice
count,1460.000000,1460.000000,1201.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1452.000000,1460.000000,...,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000
mean,730.500000,56.897260,70.049958,10516.828082,6.099315,5.575342,1971.267808,1984.865753,103.685262,443.639726,...,94.244521,46.660274,21.954110,3.409589,15.060959,2.758904,43.489041,6.321918,2007.815753,180921.195890
std,421.610009,42.300571,24.284752,9981.264932,1.382997,1.112799,30.202904,20.645407,181.066207,456.098091,...,125.338794,66.256028,61.119149,29.317331,55.757415,40.177307,496.123024,2.703626,1.328095,79442.502883
min,1.000000,20.000000,21.000000,1300.000000,1.000000,1.000000,1872.000000,1950.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,2006.000000,34900.000000
25%,365.750000,20.000000,59.000000,7553.500000,5.000000,5.000000,1954.000000,1967.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,5.000000,2007.000000,129975.000000
50%,730.500000,50.000000,69.000000,9478.500000,6.000000,5.000000,1973.000000,1994.000000,0.000000,383.500000,...,0.000000,25.000000,0.000000,0.000000,0.000000,0.000000,0.000000,6.000000,2008.000000,163000.000000
75%,1095.250000,70.000000,80.000000,11601.500000,7.000000,6.000000,2000.000000,2004.000000,166.000000,712.250000,...,168.000000,68.000000,0.000000,0.000000,0.000000,0.000000,0.000000,8.000000,2009.000000,214000.000000
max,1460.000000,190.000000,313.000000,215245.000000,10.000000,9.000000,2010.000000,2010.000000,1600.000000,5644.000000,...,857.000000,547.000000,552.000000,508.000000,480.000000,738.000000,15500.000000,12.000000,2010.000000,755000.000000


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   Alley          91 non-null     object 
 7   LotShape       1460 non-null   object 
 8   LandContour    1460 non-null   object 
 9   Utilities      1460 non-null   object 
 10  LotConfig      1460 non-null   object 
 11  LandSlope      1460 non-null   object 
 12  Neighborhood   1460 non-null   object 
 13  Condition1     1460 non-null   object 
 14  Condition2     1460 non-null   object 
 15  BldgType       1460 non-null   object 
 16  HouseStyle     1460 non-null   object 
 17  OverallQual    1460 non-null   int64  
 18  OverallC

In [6]:
df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [7]:
df = df.drop(["Id","PoolQC","MiscFeature","Alley","Fence"], axis=1)

In [8]:
num_cols = df.select_dtypes(include=['int64','float64']).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

In [9]:
cat_cols = df.select_dtypes(include='object').columns
df[cat_cols] = df[cat_cols].fillna(df[cat_cols].mode().iloc[0])

In [10]:
df = pd.get_dummies(df,drop_first=True)

In [11]:
df.head()

,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,BsmtFinSF2,...,SaleType_ConLI,SaleType_ConLw,SaleType_New,SaleType_Oth,SaleType_WD,SaleCondition_AdjLand,SaleCondition_Alloca,SaleCondition_Family,SaleCondition_Normal,SaleCondition_Partial
0,60,65.0,8450,7,5,2003,2003,196.0,706,0,...,False,False,False,False,True,False,False,False,True,False
1,20,80.0,9600,6,8,1976,1976,0.0,978,0,...,False,False,False,False,True,False,False,False,True,False
2,60,68.0,11250,7,5,2001,2002,162.0,486,0,...,False,False,False,False,True,False,False,False,True,False
3,70,60.0,9550,7,5,1915,1970,0.0,216,0,...,False,False,False,False,True,False,False,False,False,False
4,60,84.0,14260,8,5,2000,2000,350.0,655,0,...,False,False,False,False,True,False,False,False,True,False


In [12]:
X = df.drop('SalePrice',axis=1)
y=df['SalePrice']

In [13]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [19]:
model1 = AdaBoostRegressor(n_estimators = 100,random_state=42)
model1.fit(X_train,y_train)
y_pred1 = model1.predict(X_test)
print('MSE:',mean_squared_error(y_test,y_pred1))
print('MAE:',mean_absolute_error(y_test,y_pred1))
print('r2_score:',r2_score(y_test,y_pred1))

MSE: 1332081693.3728838
MAE: 26008.802167358866
r2_score: 0.8263331105625314


In [20]:
model2 = GradientBoostingRegressor()
model2.fit(X_train,y_train)
y_pred2 = model2.predict(X_test)
print('MSE:',mean_squared_error(y_test,y_pred2))
print('MAE:',mean_absolute_error(y_test,y_pred2))
print('r2_score:',r2_score(y_test,y_pred2))

MSE: 792834233.385639
MAE: 17448.870333297345
r2_score: 0.8966361779186457


In [21]:
model3 = XGBRegressor()
model3.fit(X_train,y_train)
y_pred3 = model3.predict(X_test)
print('MSE:',mean_squared_error(y_test,y_pred3))
print('MAE:',mean_absolute_error(y_test,y_pred3))
print('r2_score:',r2_score(y_test,y_pred3))

MSE: 797598016.0
MAE: 17800.61328125
r2_score: 0.8960151076316833


In [22]:
model4 = CatBoostRegressor()
model4.fit(X_train,y_train)
y_pred4 = model4.predict(X_test)
print('MSE:',mean_squared_error(y_test,y_pred4))
print('MAE:',mean_absolute_error(y_test,y_pred4))
print('r2_score:',r2_score(y_test,y_pred4))

Learning rate set to 0.04196
0:	learn: 75129.1984324	total: 61.7ms	remaining: 1m 1s
1:	learn: 73159.1219450	total: 69ms	remaining: 34.5s
2:	learn: 71258.0485182	total: 73.7ms	remaining: 24.5s
3:	learn: 69343.2529977	total: 78ms	remaining: 19.4s
4:	learn: 67668.1150813	total: 80.9ms	remaining: 16.1s
5:	learn: 66065.4920636	total: 83.9ms	remaining: 13.9s
6:	learn: 64523.6071172	total: 88.5ms	remaining: 12.6s
7:	learn: 63132.8415304	total: 91.6ms	remaining: 11.4s
8:	learn: 61614.1800154	total: 94ms	remaining: 10.4s
9:	learn: 60198.6765755	total: 97.1ms	remaining: 9.61s
10:	learn: 58765.5077034	total: 99.7ms	remaining: 8.96s
11:	learn: 57365.7230566	total: 103ms	remaining: 8.45s
12:	learn: 56047.6797528	total: 105ms	remaining: 8.01s
13:	learn: 54681.9022145	total: 108ms	remaining: 7.63s
14:	learn: 53445.7478145	total: 110ms	remaining: 7.25s
15:	learn: 52381.8434362	total: 113ms	remaining: 6.95s
16:	learn: 51350.2827522	total: 115ms	remaining: 6.65s
17:	learn: 50450.1479165	total: 117ms	rem

In [23]:
model5 = LGBMRegressor()
model5.fit(X_train,y_train)
y_pred5 = model5.predict(X_test)
print('MSE:',mean_squared_error(y_test,y_pred5))
print('MAE:',mean_absolute_error(y_test,y_pred5))
print('r2_score:',r2_score(y_test,y_pred5))

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001818 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3134
[LightGBM] [Info] Number of data points in the train set: 1168, number of used features: 151
[LightGBM] [Info] Start training from score 181441.541952
MSE: 879633055.2418954
MAE: 17068.12440668908
r2_score: 0.8853199940287186
